In [1]:
import os
import glob
import pandas as pd
import geopandas as gpd
from rasterstats import zonal_stats

# ============================================================
# INPUTS
# ============================================================

DISTRICTS = r"/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/Maps/od_ids-drr_shapefiles/odisha_district_final.geojson"

RCS = r"/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/Maps/od_ids-drr_shapefiles/odisha_block_final_reduced.geojson"

LST_FOLDER = r"/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/modis_aqua/data/state"

OUTPUT_CSV = r"/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/modis_aqua/data/variables/land_surface_temperature.csv"

# ============================================================
# READ DATA
# ============================================================

districts = gpd.read_file(DISTRICTS).to_crs("EPSG:4326")
rcs = gpd.read_file(RCS).to_crs("EPSG:4326")

DISTRICT_FIELD = "dtname"

# ============================================================
# FIND MONTHLY RASTERS
# ============================================================

rasters = sorted(
    glob.glob(
        os.path.join(
            LST_FOLDER,
            "LST_ODISHA_*.tif"
        )
    )
)

print(f"\nFound {len(rasters)} rasters")

results = []

# ============================================================
# PROCESS EACH MONTH
# ============================================================

for raster_path in rasters:

    raster_name = os.path.basename(raster_path)

    print(f"Processing {raster_name}")

    timeperiod = (
        raster_name
        .replace("LST_ODISHA_", "")
        .replace(".tif", "")
    )

    # ========================================================
    # DISTRICT MEAN LST
    # ========================================================

    stats = zonal_stats(
        districts,
        raster_path,
        stats=["mean"],
        all_touched=True
    )

    district_df = districts.copy()

    district_df["land-surface-temperature"] = [
        s["mean"] for s in stats
    ]

    # ========================================================
    # COPY DISTRICT MEAN LST TO ALL BLOCKS
    # ========================================================

    rc_month = rcs.merge(
        district_df[
            [
                DISTRICT_FIELD,
                "land-surface-temperature"
            ]
        ],
        on=DISTRICT_FIELD,
        how="left"
    )

    rc_month["timeperiod"] = timeperiod

    results.append(
        rc_month[
            [
                "object_id",
                "timeperiod",
                "land-surface-temperature"
            ]
        ]
    )

# ============================================================
# COMBINE ALL MONTHS
# ============================================================

final_df = pd.concat(
    results,
    ignore_index=True
)

# ============================================================
# SAVE
# ============================================================

final_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:")
print(OUTPUT_CSV)

print("\nRows:", len(final_df))

print(final_df.head())


Found 66 rasters
Processing LST_ODISHA_2021_01.tif


/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/.venv/lib/python3.12/site-packages/rasterstats/io.py:437: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Processing LST_ODISHA_2021_02.tif
Processing LST_ODISHA_2021_03.tif
Processing LST_ODISHA_2021_04.tif
Processing LST_ODISHA_2021_05.tif
Processing LST_ODISHA_2021_06.tif
Processing LST_ODISHA_2021_07.tif
Processing LST_ODISHA_2021_08.tif
Processing LST_ODISHA_2021_09.tif
Processing LST_ODISHA_2021_10.tif
Processing LST_ODISHA_2021_11.tif
Processing LST_ODISHA_2021_12.tif
Processing LST_ODISHA_2022_01.tif
Processing LST_ODISHA_2022_02.tif
Processing LST_ODISHA_2022_03.tif
Processing LST_ODISHA_2022_04.tif
Processing LST_ODISHA_2022_05.tif
Processing LST_ODISHA_2022_06.tif
Processing LST_ODISHA_2022_07.tif
Processing LST_ODISHA_2022_08.tif
Processing LST_ODISHA_2022_09.tif
Processing LST_ODISHA_2022_10.tif
Processing LST_ODISHA_2022_11.tif
Processing LST_ODISHA_2022_12.tif
Processing LST_ODISHA_2023_01.tif
Processing LST_ODISHA_2023_02.tif
Processing LST_ODISHA_2023_03.tif
Processing LST_ODISHA_2023_04.tif
Processing LST_ODISHA_2023_05.tif
Processing LST_ODISHA_2023_06.tif
Processing LST